---
## Setup: Start vLLM + Onboard OpenClaw + Start Gateway

Run the cell below **once**. It will:
1. Start vLLM serving Qwen3-4B on your AMD GPU (background)
2. Tail the startup logs live until the model is loaded
3. Onboard OpenClaw against the local vLLM endpoint
4. Activate the credential and set lean mode
5. Start the OpenClaw gateway (so you can chat from the notebook)

When it finishes (green), proceed below to chat with your OpenClaw agent directly in this notebook.

In [ ]:
import subprocess, time, os, re, sys
try:
    import httpx
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "httpx"])
    import httpx

# ── 1. Start vLLM in background ─────────────────────────────────────
print("▶ Starting vLLM server...")
env = {**os.environ, "VLLM_USE_TRITON_FLASH_ATTN": "0"}
vllm_proc = subprocess.Popen(
    ["vllm", "serve", "Qwen/Qwen3-4B",
     "--served-model-name", "Qwen3-4B",
     "--api-key", "abc-123",
     "--port", "8000",
     "--enable-auto-tool-choice",
     "--tool-call-parser", "hermes",
     "--trust-remote-code",
     "--max_model_len", "24272",
     "--gpu-memory-utilization", "0.85"],
    stdout=open("/tmp/vllm.log", "w"),
    stderr=subprocess.STDOUT,
    env=env,
)
print(f"  PID {vllm_proc.pid} launched, waiting for model load...\n")

# ── 2. Tail vLLM log + poll /health until ready ─────────────────────
seen = 0
ready = False
for tick in range(300):  # up to ~5 min
    time.sleep(1)
    try:
        with open("/tmp/vllm.log") as f:
            lines = f.readlines()
        new_lines = lines[seen:]
        if new_lines:
            for ln in new_lines[-5:]:  # show last 5 new lines
                print(ln.rstrip())
            seen = len(lines)
    except FileNotFoundError:
        pass
    # Check health every 5s
    if tick % 5 == 4:
        try:
            r = httpx.get("http://localhost:8000/health", timeout=2)
            if r.status_code == 200:
                ready = True
                break
        except Exception:
            pass

if not ready:
    print("\n✗ vLLM did not become healthy in 5 minutes. Check /tmp/vllm.log")
    raise SystemExit(1)

print("\n✓ vLLM is healthy!\n")

# ── 3. Onboard OpenClaw ──────────────────────────────────────────────
print("▶ Onboarding OpenClaw...")
r = subprocess.run([
    "openclaw", "onboard", "--non-interactive", "--mode", "local",
    "--auth-choice", "custom-api-key",
    "--custom-base-url", "http://localhost:8000/v1",
    "--custom-model-id", "Qwen3-4B", "--custom-provider-id", "vllm",
    "--custom-compatibility", "openai", "--custom-api-key", "abc-123",
    "--secret-input-mode", "plaintext", "--custom-image-input",
    "--skip-channels", "--skip-search", "--skip-hooks", "--skip-ui",
    "--skip-health", "--accept-risk",
], capture_output=True, text=True)
print(r.stdout.strip())
if r.stderr.strip():
    print(r.stderr.strip())

# Activate the latest credential
match = re.search(r"(vllm:setup-[\w-]+)", r.stdout + r.stderr)
if match:
    cred_id = match.group(1)
    print(f"\n▶ Activating credential {cred_id}...")
    a = subprocess.run(
        ["openclaw", "models", "auth", "activate", cred_id, "--agent", "main"],
        capture_output=True, text=True,
    )
    print(a.stdout.strip())

# ── 4. Lean mode config ─────────────────────────────────────────────
print("\n▶ Setting lean mode...")
subprocess.run(
    ["openclaw", "config", "set",
     "agents.defaults.experimental.localModelLean", "true"],
    capture_output=True, text=True,
)
print("  Config applied.")

# ── 5. Start OpenClaw gateway ────────────────────────────────────────
print("\n▶ Starting OpenClaw gateway...")
subprocess.Popen(
    ["openclaw", "gateway", "start"],
    stdout=open("/tmp/openclaw-gateway.log", "w"),
    stderr=subprocess.STDOUT,
)

gw_ready = False
for tick in range(60):
    time.sleep(1)
    try:
        r = httpx.get("http://localhost:8080/health", timeout=2)
        if r.status_code == 200:
            gw_ready = True
            break
    except Exception:
        pass

if gw_ready:
    print("✓ OpenClaw gateway is ready on port 8080!")
else:
    print("⚠ Gateway did not respond on :8080 within 60s.")
    print("  Check /tmp/openclaw-gateway.log — chat may still work after a moment.")

print("\n" + "=" * 60)
print("✓ All done! Run the cells below to chat with your agent.")
print("=" * 60)

---
## Chat with OpenClaw

Run the cell below to launch `openclaw tui` **inside this notebook**. A terminal will appear below — just start chatting with your agent directly. Try saying **Hi** first!

In [ ]:
import os, json, time
try:
    import requests as _req
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "requests"])
    import requests as _req
try:
    import websocket as _ws
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "websocket-client"])
    import websocket as _ws
from IPython.display import IFrame, display

# ── Discover Jupyter server details ──────────────────────────────────
base_prefix = os.environ.get("JUPYTERHUB_SERVICE_PREFIX", "/")
if not base_prefix.endswith("/"):
    base_prefix += "/"
server_port = os.environ.get("JUPYTER_PORT", "8888")
local_base = f"http://localhost:{server_port}{base_prefix}"
token = os.environ.get("JPY_API_TOKEN", os.environ.get("JUPYTER_TOKEN", ""))

headers = {}
if token:
    headers["Authorization"] = f"token {token}"

# ── Create a terminal session ────────────────────────────────────────
print("Creating terminal session...")
resp = _req.post(f"{local_base}api/terminals", headers=headers)
resp.raise_for_status()
term_name = resp.json()["name"]
print(f"  Terminal '{term_name}' created.")

# ── Send 'openclaw tui' to the terminal via WebSocket ────────────────
print("Launching openclaw tui in the terminal...")
ws_url = f"ws://localhost:{server_port}{base_prefix}terminals/websocket/{term_name}"
ws_extra = {}
if token:
    ws_extra["header"] = [f"Authorization: token {token}"]

ws = _ws.create_connection(ws_url, **ws_extra)
time.sleep(0.5)
ws.send(json.dumps(["stdin", "openclaw tui\r"]))
time.sleep(0.5)
ws.close()
print("  openclaw tui is starting up.\n")

# ── Embed the terminal in this notebook cell ─────────────────────────
iframe_url = f"{base_prefix}terminals/{term_name}"
print(f"Embedding terminal at: {iframe_url}")
display(IFrame(src=iframe_url, width="100%", height="500"))

---
## Try it: Clone a Repo and Ask for Analysis

Copy and paste the prompt below into the OpenClaw chat above:

```
Clone https://github.com/matrived/blr_workshop.git in /workspace directory, show me the repository structure line-by-line one-liner description of them and also describe cell-by-cell what unsloth-finetuning/grpo_game2048-finetune-unsloth/gpt_oss_20B_Reinforcement_Learning_2048_Game_BF16.ipynb file is doing in each cell in 1 line and lastly what could be improved
```